# Experiments to evaluate autoscaling cost

In [ ]:
# Basic imports and method to write autoscaling results
import numpy as np
from numpy import percentile as percentile
import pandas as pd
from ascal import AscalConfig, Ascal
from examples import aws_eu_west_1_c5m5r5
import os

def plot_ascal_results(ascal_problem: Ascal, exp_name: str):
    # Get application overloads as workload/performance
    workloads = ascal_problem.get_workloads()
    performances = ascal_problem.get_performances()
    overloads = {app: [(w-p)/w for w, p in zip(workloads[app], performances[app])] for app in workloads}

    # Get queue waiting times relative to service times, assuming each container is a server in a 
    # heterogenous D/D/n queue
    relative_queue_waiting_times = ascal_problem.get_relative_queue_waiting_times()
    avgs = {
        app_name: sum(waiting_times) / len(waiting_times)
        for app_name, waiting_times in relative_queue_waiting_times.items()
    }
    for app_name in dict(relative_queue_waiting_times):
        relative_queue_waiting_times[f"{app_name} avg = {avgs[app_name]:.6f}"] =\
            relative_queue_waiting_times.pop(app_name)
    # Plot autoscaling information
    ascal_problem.plot(ascal_problem.get_performances(), f"Application Performances ({exp_name})", "req/s")
    cluster_cost = ascal_problem.get_cluster_cost()
    total_cost_str = f"total cost = {sum(cluster_cost)/3600:.3f} $"
    ascal_problem.plot({total_cost_str: cluster_cost}, f"Cluster Cost ({exp_name})", "$/hour")
    ascal_problem.plot(overloads, f"Application Overloads ({exp_name})")
    ascal_problem.plot(relative_queue_waiting_times, f"Relative queue waiting times ({exp_name})")

## Generate the trace files
The trace file has 3600 samples. Note that the number of samples can be multiplied using `time_interval` field in 
ASCAL configuration files.

In [ ]:
num_samples = 3600

# Generate a trapezoidal waveform with random noise and save it to a CSV file
min_val = 10
max_val = 100
noise_amplitude = 0.1

# Generate complete trapezoidal waveform (ramp up, plateau, ramp down)
ramp_len = (num_samples - 1800) // 2
ramp_up = np.linspace(min_val, max_val, ramp_len)
plateau = np.full(1800, max_val)
ramp_down = np.linspace(max_val, min_val, ramp_len)
trapezoid = np.concatenate([ramp_up, plateau, ramp_down])
trapezoid = trapezoid[:num_samples]

# Add random noise
np.random.seed(0)  # For reproducibility
noise = np.random.normal(0, noise_amplitude * (trapezoid + 20))
trapezoid_noisy = trapezoid + noise
trapezoid_noisy = np.round(trapezoid_noisy).astype(int)

# Create DataFrame and save to CSV
df_triangle = pd.DataFrame({'requests/s': trapezoid_noisy})
df_triangle.to_csv('trapezoid.csv', index=False)

## HPA+CA

In [ ]:
# Set experiment
config_file = "exp-hpa-ca.yaml"
ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
ascal_problem = Ascal(ascal_config) 
ascal_problem.plot(ascal_problem.get_workloads(), "Application Workloads", "req/s")

In [ ]:
# Run experiment
ascal_problem.run()

In [ ]:
# Plot results
plot_ascal_results(ascal_problem, "HPA+CA")

## HV Reactive FCMA

In [ ]:
# Set experiment
config_file = "exp-hv-reactive-fcma.yaml"
ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
ascal_problem = Ascal(ascal_config)

In [ ]:
# Run experiment
ascal_problem.run()

In [ ]:
# Plot results
plot_ascal_results(ascal_problem, "HV Reactive FCMA")

## HV Predictive FCMA

In [ ]:
# Set experiment
config_file = "exp-hv-predictive-fcma.yaml"
ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
ascal_problem = Ascal(ascal_config)

In [ ]:
# Run experiment
ascal_problem.run()

In [ ]:
# Plot results
plot_ascal_results(ascal_problem, "HV Predictive FCMA")

## HPA+CA+HV Reactive FCMA

In [ ]:
# Set experiment
config_file = "exp-hpa-ca-hv-reactive-fcma.yaml"
ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
ascal_problem = Ascal(ascal_config)

In [ ]:
# Run experiment
ascal_problem.run()

In [ ]:
# Plot results
plot_ascal_results(ascal_problem, "HPA+CA+HV Reactive FCMA")

## HPA+CA+HV Predictive FCMA

In [ ]:
# Set experiment
config_file = "exp-hpa-ca-hv-predictive-fcma.yaml"
ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
ascal_problem = Ascal(ascal_config)

In [ ]:
# Run experiment
ascal_problem.run()

In [ ]:
# Plot results
plot_ascal_results(ascal_problem, "HPA+CA+HV Predictive FCMA")